<img src="images/Exercise.png" width="500">

In [15]:
from cmath import sqrt as complex_sqrt
from math import sqrt


In [16]:
def cal_discriminant(a,b,c):
    return b**2-4*a*c

In [20]:
a=1
b=-5
c=6

discriminant = cal_discriminant(a,b,c)

if discriminant<0:
     discriminant = complex_sqrt(discriminant)
else:
    discriminant = sqrt(discriminant)

x=(-b+discriminant)/(2*a)
y=(-b-discriminant)/(2*a)
print(x,y)


3.0 2.0


## No — Two Real Bugs Here, Both Worth Understanding

Let's trace through and see what actually breaks.

---

### Bug 1 — `i` Is Not Defined

```python
discriminant = i*sqrt(-discriminant)
```

You're trying to write the imaginary unit — but Python uses **`j`**, not `i`, and even then, `j` alone isn't valid either (remember from the `complex` lesson: it needs to be attached directly to a number, like `1j`, not used as a standalone variable):

```python
i*sqrt(-discriminant)
# ✗ NameError: name 'i' is not defined
```

`i` here is being read as an **undefined variable name** — Python has no idea what `i` refers to, because you never assigned it.

---

### Bug 2 — Even With `1j`, `sqrt()` from `math` Can't Handle Negative Numbers

```python
from math import sqrt
sqrt(-9)
# ✗ ValueError: math domain error
```

`math.sqrt()` is built for **real numbers only** — it explicitly refuses negative inputs, since (in the world of real numbers) they have no defined square root. This is exactly **why** complex numbers exist — remember your `complex` lesson: *"complex numbers exist specifically to let you take the square root of negative numbers, which ordinary real numbers can't do."*

You need `cmath.sqrt()` instead — the complex-number version:

```python
from cmath import sqrt
sqrt(-9)     # → 3j      ✓ works! (returns a complex number)
```

---

### Trace Through Your Actual Example — Does It Even Hit This Case?

```python
a, b, c = 2, -3, 2

discriminant = cal_discriminant(a, b, c)
# = (-3)² - 4(2)(2) = 9 - 16 = -7
```

**Yes — with `a=2, b=-3, c=2`, the discriminant is `-7`, which is negative** — so your code **does** reach the buggy `i*sqrt(...)` line, and would crash immediately on `NameError: name 'i' is not defined`.

---

### The Fix — Use `cmath`

```python
from cmath import sqrt


def cal_discriminant(a, b, c):
    return b**2 - 4*a*c


a, b, c = 2, -3, 2

discriminant = cal_discriminant(a, b, c)
discriminant = sqrt(discriminant)      # cmath.sqrt handles BOTH positive and negative!

x = (-b + discriminant) / (2*a)
y = (-b - discriminant) / (2*a)
print(x, y)
```

**Key simplification:** `cmath.sqrt()` handles **both** positive and negative numbers correctly — no need for the `if discriminant < 0:` branch at all! For a positive discriminant, it just returns a normal-looking complex number with `0j` imaginary part:

```python
from cmath import sqrt
sqrt(25)      # → (5+0j)     positive input, still works fine
sqrt(-7)       # → 2.6457513110645907j     negative input, now handled!
```

---

### Testing It

```python
print(x, y)
# → (0.75+1.3228756555322954j) (0.75-1.3228756555322954j)
```

Two **complex conjugate** roots — exactly what you'd expect mathematically, since a negative discriminant means the quadratic has no *real* solutions, only complex ones.

---

### If You WANT to Keep Real/Complex Cases Separate

If your intent was specifically to use `math.sqrt` for the simple real case and only switch to complex math when needed (perhaps for learning purposes), the corrected version would look like:

```python
import math
import cmath


def cal_discriminant(a, b, c):
    return b**2 - 4*a*c


a, b, c = 2, -3, 2

discriminant = cal_discriminant(a, b, c)

if discriminant < 0:
    sqrt_d = cmath.sqrt(discriminant)     # complex result — use cmath
else:
    sqrt_d = math.sqrt(discriminant)       # real result — math is fine

x = (-b + sqrt_d) / (2*a)
y = (-b - sqrt_d) / (2*a)
print(x, y)
```

Both versions are correct — the `cmath`-only version is simpler (no branching needed), while this version keeps `math.sqrt` for the common real-number case and only reaches for `cmath` when actually necessary.

---

### The Two Bugs, Summarized

| Bug | Problem | Fix |
|---|---|---|
| `i*sqrt(...)` | `i` is undefined — Python uses `j`, and even that needs to attach to a number (`1j`) | Don't multiply by `i`/`j` manually — let `cmath.sqrt()` return the complex result directly |
| `from math import sqrt` | `math.sqrt` crashes on negative input | Use `from cmath import sqrt` instead — handles negatives natively |

---

### The One-Sentence Summary

> No — this crashes with `NameError: name 'i' is not defined`, because Python's imaginary unit is written as `j` (attached to a number, like `1j`), not a standalone `i`; and even fixing that, `math.sqrt()` specifically refuses negative numbers — you need `cmath.sqrt()`, which natively returns complex results and actually eliminates the need for the `if discriminant < 0:` branch entirely. 🎯